In [ ]:
pip install qiskit qiskit-machine-learning scikit-learn qiskit-algorithms qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 87.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from tensorflow.keras.datasets import mnist, fashion_mnist

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_aer import Aer

LOAD AND PRE-PROCESS DATA

In [ ]:
(train_X, train_y), (test_X, test_y) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Filter for binary classification (Digits 0 and 1)

In [ ]:
train_mask = np.isin(train_y, [0, 1])
test_mask = np.isin(test_y, [0, 1])

In [ ]:
X_train, y_train = train_X[train_mask][:100], train_y[train_mask][:100]
X_test, y_test = test_X[test_mask][:20], test_y[test_mask][:20]

Flatten and Normalize

In [ ]:
X_train = X_train.reshape(-1, 784) / 255.0
X_test = X_test.reshape(-1, 784) / 255.0

DIMENSIONALITY REDUCTION (784 PIXCELS = 4 FEATURES)

In [ ]:
pca = PCA(n_components=4)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

CONSTRUCT QUANTUM KERNEL

In [ ]:
num_qubits = 4
feature_map = ZZFeatureMap(feature_dimension=num_qubits, reps=2, entanglement='linear')

/tmp/ipykernel_1133/225157481.py:2: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=num_qubits, reps=2, entanglement='linear')


In [ ]:
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms.state_fidelities import ComputeUncompute

# 1. Initialize the V2 Sampler
sampler = StatevectorSampler()

# 2. Pass the sampler instance to ComputeUncompute
fidelity = ComputeUncompute(sampler=sampler)

# 3. Proceed with the Kernel
quantum_kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)

TRAIN CLASSICAL SVM WITH QUANTUM KERNEL

In [ ]:
matrix_train = quantum_kernel.evaluate(x_vec=X_train_pca)
matrix_test = quantum_kernel.evaluate(x_vec=X_test_pca, y_vec=X_train_pca)

In [ ]:
qsvm = SVC(kernel='precomputed')
qsvm.fit(matrix_train, y_train)

SVC(kernel='precomputed')

In [ ]:
score = qsvm.score(matrix_test, y_test)
print(f"QSVM Classification Accuracy: {score * 100}%")

QSVM Classification Accuracy: 50.0%
